In [8]:
import pandas as pd

pairs_df = pd.read_csv('simulados_pairs.csv')
pairs_df.head()

,aluno_id,prev_created_at,prev_matematica,prev_linguagens,prev_humanas,prev_natureza,next_matematica,next_linguagens,next_humanas,next_natureza
0,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-02 18:01:44.226071+00:00,25.166667,29.833333,31.833333,25.0,25.0,NaN,NaN,21.0
1,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-28 01:32:38.720997+00:00,25.000000,NaN,NaN,21.0,24.0,NaN,NaN,NaN
2,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-05-06 20:37:03.468013+00:00,24.000000,NaN,NaN,NaN,24.0,27.0,33.0,25.0
3,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-06-08 23:10:21.903540+00:00,24.000000,27.000000,33.000000,25.0,31.0,23.0,38.0,25.0
4,1685cd68-0db4-41e2-84f7-5065022bb134,2026-04-02 18:23:16.513865+00:00,41.000000,40.000000,40.000000,39.0,35.0,33.0,35.0,31.0


In [9]:
# Cell 2 — add tests_count feature

# for each student, number their pairs 1, 2, 3...
# cumcount() counts rows within each group, starting at 0, so we add 1
pairs_df['tests_count'] = pairs_df.groupby('aluno_id').cumcount() + 1

# verify: every student should start at 1
pairs_df.groupby('aluno_id')['tests_count'].min()

aluno_id
05eebd85-ad17-4d43-934f-1cbce6cf446f    1
1685cd68-0db4-41e2-84f7-5065022bb134    1
2119aaa5-f0a9-4cb8-bc30-3a5504efd77d    1
233afb42-cb42-41fe-a7fc-e865bc9615f1    1
2a136dad-e4c5-4de8-b4b1-fb18788a8f44    1
2d95dc73-c750-4806-a7cf-0deb07faad30    1
3e34a61b-98e0-48c8-951e-c9cf83a78c41    1
41180d34-1434-4ee9-88f1-d12766ceb207    1
4d3ee5f1-21e1-444d-94fd-d3d0832f0ea6    1
4e8115b5-3e5b-439c-867b-a876befcdea4    1
53f74b3a-3973-4af8-a567-55ce939243ac    1
58d56284-c5de-4e5a-beff-b81975b6c09a    1
5a37ed4e-7ba3-41e9-86e2-aff4f4b021e1    1
5aa85d64-c555-428b-bd7f-a226396a05dc    1
5b152cab-3b5e-43c2-8491-f5c88ef7d9f5    1
64936158-5ae8-48cf-b5cd-c6c4d0aabcd9    1
724a4bd8-fd21-401a-bf6c-4bb541054faa    1
8089b383-82f8-4465-a2f0-89e8f1841d24    1
8b7fa671-82e2-4a57-90d8-cbd0b92001f8    1
9c09a49b-5fee-48a1-9710-64f74255b56d    1
9efd46a8-b99a-4474-9d41-1a7fc6ba63a4    1
ae2ab9aa-c61d-4df0-9df7-b8b74b841471    1
b98b036c-56cf-4167-932b-4e8cafb4218d    1
d67f8842-31bb-421d-8b0e-0

In [10]:
# Cell 3 — load the full test history (needed to compute trend and rolling mean)

final_df = pd.read_csv('simulados_paired_ready.csv')
final_df['created_at'] = pd.to_datetime(final_df['created_at'])
final_df = final_df.sort_values(['aluno_id', 'created_at']).reset_index(drop=True)
print(len(final_df))
final_df.head()

148


,aluno_id,created_at,matematica,linguagens,humanas,natureza
0,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-02 18:01:44.226071+00:00,25.166667,29.833333,31.833333,25.0
1,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-28 01:32:38.720997+00:00,25.000000,NaN,NaN,21.0
2,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-05-06 20:37:03.468013+00:00,24.000000,NaN,NaN,NaN
3,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-06-08 23:10:21.903540+00:00,24.000000,27.000000,33.000000,25.0
4,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-06-08 23:17:19.204186+00:00,31.000000,23.000000,38.000000,25.0


In [11]:
# Cell 4 — compute trend (slope of scores over time) per student per area

import numpy as np

areas = ['matematica', 'linguagens', 'humanas', 'natureza']

def compute_trend(scores):
    # scores is a series of values in chronological order
    # we need at least 2 points to compute a slope
    clean = scores.dropna()
    if len(clean) < 2:
        return np.nan  # can't compute slope from a single point
    
    # np.polyfit(x, y, 1) fits a straight line to the data
    # x is just position (0, 1, 2...), y is the actual scores
    # [0] pulls out just the slope, ignoring the intercept
    return np.polyfit(range(len(clean)), clean.values, 1)[0]

# for each area, compute the expanding trend per student
# "expanding" means: at row N, use all rows from 0 to N (cumulative, not fixed window)
for area in areas:
    trend_col = f'trend_{area}'
    final_df[trend_col] = (
        final_df.groupby('aluno_id')[area]
        .expanding()
        .apply(compute_trend, raw=False)
        .reset_index(level=0, drop=True)  # expanding() adds an extra index layer, this removes it
    )

final_df[['aluno_id', 'created_at'] + areas + [f'trend_{area}' for area in areas]].head(20)

,aluno_id,created_at,matematica,linguagens,humanas,natureza,trend_matematica,trend_linguagens,trend_humanas,trend_natureza
0,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-02 18:01:44.226071+00:00,25.166667,29.833333,31.833333,25.0,NaN,NaN,NaN,NaN
1,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-28 01:32:38.720997+00:00,25.000000,NaN,NaN,21.0,-0.166667,NaN,NaN,-4.000000e+00
2,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-05-06 20:37:03.468013+00:00,24.000000,NaN,NaN,NaN,-0.583333,NaN,NaN,-4.000000e+00
3,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-06-08 23:10:21.903540+00:00,24.000000,27.000000,33.000000,25.0,-0.450000,-2.833333e+00,1.166667,-1.522248e-15
4,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-06-08 23:17:19.204186+00:00,31.000000,23.000000,38.000000,25.0,1.066667,-3.416667e+00,3.083333,4.000000e-01
5,1685cd68-0db4-41e2-84f7-5065022bb134,2026-04-02 18:23:16.513865+00:00,41.000000,40.000000,40.000000,39.0,NaN,NaN,NaN,NaN
6,1685cd68-0db4-41e2-84f7-5065022bb134,2026-04-16 03:14:41.350139+00:00,35.000000,33.000000,35.000000,31.0,-6.000000,-7.000000e+00,-5.000000,-8.000000e+00
7,1685cd68-0db4-41e2-84f7-5065022bb134,2026-04-16 03:14:58.045098+00:00,29.000000,28.000000,35.000000,20.0,-6.000000,-6.000000e+00,-2.500000,-9.500000e+00
8,1685cd68-0db4-41e2-84f7-5065022bb134,2026-04-16 14:00:21.149869+00:00,42.000000,39.000000,38.000000,44.0,-0.300000,-8.000000e-01,-0.600000,4.000000e-01
9,1685cd68-0db4-41e2-84f7-5065022bb134,2026-04-16 14:00:56.863694+00:00,43.000000,39.000000,41.000000,42.0,1.100000,4.000000e-01,0.500000,1.900000e+00


In [12]:
# Cell 5 — compute rolling mean (average of last 3 scores) per student per area

for area in areas:
    rolling_col = f'rolling_mean_{area}'
    final_df[rolling_col] = (
        final_df.groupby('aluno_id')[area]
        .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
        # min_periods=1 means: if fewer than 3 scores exist yet, 
        # average whatever is available rather than returning NaN
        # so a student's first test gets rolling_mean = that score itself
        # second test gets average of first two, third onwards gets average of last 3
    )

final_df[['aluno_id'] + areas + [f'rolling_mean_{area}' for area in areas]].head(20)

,aluno_id,matematica,linguagens,humanas,natureza,rolling_mean_matematica,rolling_mean_linguagens,rolling_mean_humanas,rolling_mean_natureza
0,05eebd85-ad17-4d43-934f-1cbce6cf446f,25.166667,29.833333,31.833333,25.0,25.166667,29.833333,31.833333,25.000000
1,05eebd85-ad17-4d43-934f-1cbce6cf446f,25.000000,NaN,NaN,21.0,25.083333,29.833333,31.833333,23.000000
2,05eebd85-ad17-4d43-934f-1cbce6cf446f,24.000000,NaN,NaN,NaN,24.722222,29.833333,31.833333,23.000000
3,05eebd85-ad17-4d43-934f-1cbce6cf446f,24.000000,27.000000,33.000000,25.0,24.333333,27.000000,33.000000,23.000000
4,05eebd85-ad17-4d43-934f-1cbce6cf446f,31.000000,23.000000,38.000000,25.0,26.333333,25.000000,35.500000,25.000000
5,1685cd68-0db4-41e2-84f7-5065022bb134,41.000000,40.000000,40.000000,39.0,41.000000,40.000000,40.000000,39.000000
6,1685cd68-0db4-41e2-84f7-5065022bb134,35.000000,33.000000,35.000000,31.0,38.000000,36.500000,37.500000,35.000000
7,1685cd68-0db4-41e2-84f7-5065022bb134,29.000000,28.000000,35.000000,20.0,35.000000,33.666667,36.666667,30.000000
8,1685cd68-0db4-41e2-84f7-5065022bb134,42.000000,39.000000,38.000000,44.0,35.333333,33.333333,36.000000,31.666667
9,1685cd68-0db4-41e2-84f7-5065022bb134,43.000000,39.000000,41.000000,42.0,38.000000,35.333333,38.000000,35.333333


In [16]:
# Cell 6 — fix datetime types so the merge keys match
pairs_df['prev_created_at'] = pd.to_datetime(pairs_df['prev_created_at'], utc=True)
final_df['created_at'] = pd.to_datetime(final_df['created_at'], utc=True)

# Cell 6 — bring trend and rolling mean columns onto pairs_df

# the columns we want to carry over from final_df
feature_cols = (
    ['aluno_id', 'created_at'] +
    [f'trend_{area}' for area in areas] +
    [f'rolling_mean_{area}' for area in areas]
)

# merge pairs_df with final_df on aluno_id + the prev test's timestamp
# this lines up each pair row with the exact moment of the prev test
pairs_df = pairs_df.merge(
    final_df[feature_cols],
    left_on=['aluno_id', 'prev_created_at'],
    right_on=['aluno_id', 'created_at'],
    how='left'
).drop(columns='created_at')  # created_at is redundant after merge, prev_created_at already has it

pairs_df.head()

,aluno_id,prev_created_at,prev_matematica,prev_linguagens,prev_humanas,prev_natureza,next_matematica,next_linguagens,next_humanas,next_natureza,...,rolling_mean_humanas_x,rolling_mean_natureza_x,trend_matematica_y,trend_linguagens_y,trend_humanas_y,trend_natureza_y,rolling_mean_matematica_y,rolling_mean_linguagens_y,rolling_mean_humanas_y,rolling_mean_natureza_y
0,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-02 18:01:44.226071+00:00,25.166667,29.833333,31.833333,25.0,25.0,NaN,NaN,21.0,...,31.833333,25.0,NaN,NaN,NaN,NaN,25.166667,29.833333,31.833333,25.0
1,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-04-28 01:32:38.720997+00:00,25.000000,NaN,NaN,21.0,24.0,NaN,NaN,NaN,...,31.833333,23.0,-0.166667,NaN,NaN,-4.000000e+00,25.083333,29.833333,31.833333,23.0
2,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-05-06 20:37:03.468013+00:00,24.000000,NaN,NaN,NaN,24.0,27.0,33.0,25.0,...,31.833333,23.0,-0.583333,NaN,NaN,-4.000000e+00,24.722222,29.833333,31.833333,23.0
3,05eebd85-ad17-4d43-934f-1cbce6cf446f,2026-06-08 23:10:21.903540+00:00,24.000000,27.000000,33.000000,25.0,31.0,23.0,38.0,25.0,...,33.000000,23.0,-0.450000,-2.833333,1.166667,-1.522248e-15,24.333333,27.000000,33.000000,23.0
4,1685cd68-0db4-41e2-84f7-5065022bb134,2026-04-02 18:23:16.513865+00:00,41.000000,40.000000,40.000000,39.0,35.0,33.0,35.0,31.0,...,40.000000,39.0,NaN,NaN,NaN,NaN,41.000000,40.000000,40.000000,39.0


In [17]:
# Cell 7 — save enriched pairs with all features
pairs_df.to_csv('simulados_pairs_features.csv', index=False)
print(f"Saved {len(pairs_df)} rows with {len(pairs_df.columns)} columns")
print(pairs_df.columns.tolist())

Saved 115 rows with 27 columns
['aluno_id', 'prev_created_at', 'prev_matematica', 'prev_linguagens', 'prev_humanas', 'prev_natureza', 'next_matematica', 'next_linguagens', 'next_humanas', 'next_natureza', 'tests_count', 'trend_matematica_x', 'trend_linguagens_x', 'trend_humanas_x', 'trend_natureza_x', 'rolling_mean_matematica_x', 'rolling_mean_linguagens_x', 'rolling_mean_humanas_x', 'rolling_mean_natureza_x', 'trend_matematica_y', 'trend_linguagens_y', 'trend_humanas_y', 'trend_natureza_y', 'rolling_mean_matematica_y', 'rolling_mean_linguagens_y', 'rolling_mean_humanas_y', 'rolling_mean_natureza_y']
